In [2]:
import pandas as pd
import numpy as np
import os
import ast
from tqdm import tqdm
tqdm.pandas()
from collections import Counter

In [ ]:
ls_file_nm = os.listdir('../preprocessed_data/')[-3:]

df = pd.DataFrame()
for val_file in ls_file_nm:
    df_sub = pd.read_csv(f'../preprocessed_data/{val_file}')
    df_sub['media'] = val_file.split('_')[1]

    df = pd.concat([df, df_sub])
df.head(1)

In [4]:
df['tokens'] = df['tokens'].progress_apply(ast.literal_eval)

100%|██████████████████████████████████████████████████████████████████████████| 35299/35299 [00:27<00:00, 1285.31it/s]


In [5]:
df_msg_cnt_bind = pd.DataFrame()

for val_date in df['date'].unique():
    ser_msg = df.loc[df['date'].isin([val_date]), 'tokens']
    ser_msg = ser_msg.explode()
    ser_msg_cnt = ser_msg.value_counts()
    ser_msg_cnt.index.name = "word"
    df_msg_cnt = ser_msg_cnt.reset_index()
    df_msg_cnt['date'] = val_date

    df_msg_cnt_bind = pd.concat([df_msg_cnt_bind, df_msg_cnt])

In [6]:
df_msg_cnt_bind['year'] = df_msg_cnt_bind['date'].apply(lambda x: int(str(x)[:4]))   
df_msg_cnt_bind['month'] = df_msg_cnt_bind['date'].apply(lambda x: int(str(x)[-2:]))

In [7]:
for year in df_msg_cnt_bind['year'].unique():

    print('\n', '='*20, '\n')
    print(f'{year}년 빈도 상위 Top10', '\n')
    print(df_msg_cnt_bind.loc[df_msg_cnt_bind['year'].isin([year]),['word', 'count']].head(10))



2016년 빈도 상위 Top10 

  word  count
0   사회    826
1   정부    718
2   경제    577
3   사람    555
4   취업    517
5   기업    496
6   한국    492
7   문제    463
8   지원    437
9   정치    427


2017년 빈도 상위 Top10 

  word  count
0  대통령    503
1  일자리    489
2   경제    440
3   대표    438
4   정부    430
5   사회    392
6   사람    364
7   정치    352
8   기업    332
9  지난해    325


2018년 빈도 상위 Top10 

  word  count
0   정부    707
1   사회    582
2  일자리    582
3  대통령    466
4   정책    463
5  지난해    429
6   사람    400
7   한국    364
8   기업    363
9   문제    359


2019년 빈도 상위 Top10 

  word  count
0   사회    666
1   정부    639
2   사람    496
3   경제    464
4   기업    458
5   임금    376
6   서울    360
7  대통령    359
8   지역    358
9  지난해    345


2020년 빈도 상위 Top10 

  word  count
0   대표    499
1   사회    482
2   정치    442
3   한국    429
4   정부    351
5   의원    344
6   기업    336
7   서울    333
8   문제    328
9  지난해    324


2021년 빈도 상위 Top10 

  word  count
0   사회    425
1   정부    366
2  코로나    356
3   지원    334
4  지난해    323
5  일자리    314


### 공출현 행렬 구축 - 문장단위

In [ ]:
import sys, os, pickle
from pathlib import Path
from collections import Counter
from itertools import combinations

import pandas as pd
from tqdm import tqdm
from kiwipiepy import Kiwi
import networkx as nx

# 프로젝트 루트: 노트북이 02_network_analysis/ 하위에 있으므로 부모 디렉토리를 사용
_cwd = Path(os.getcwd())
_project_root = _cwd.parent
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_cwd))  # synonym_stopword.py는 같은 디렉토리

output_dir = _project_root / "output"
output_dir.mkdir(exist_ok=True)

from synonym_stopword import USER_DICT, STOPWORDS, preprocess_1st, preprocess_2nd

kiwi = Kiwi()
for word, tag in USER_DICT:
    kiwi.add_user_word(word, tag)

print(f"프로젝트 루트: {_project_root}")
print(f"출력 디렉토리: {output_dir}")
print(f"기사 수: {len(df):,}건")
print(f"STOPWORDS 크기: {len(STOPWORDS)}개")

In [ ]:
def sent_tokenize(text_cleaned: str) -> list:
    """기사 text_cleaned → 문장별 명사 토큰 리스트"""
    text = preprocess_1st(text_cleaned)
    sents = kiwi.split_into_sents(text, return_tokens=False)
    result = []
    for sent in sents:
        raw_tokens = [
            t.form for t in kiwi.tokenize(sent.text)
            if t.tag in ("NNG", "NNP") and len(t.form) > 1
        ]
        cleaned = preprocess_2nd(raw_tokens)
        if len(cleaned) >= 2:
            result.append(cleaned)
    return result

CACHE_PATH = _project_root / "output" / "sent_tokens_cache.pkl"

if CACHE_PATH.exists():
    with open(CACHE_PATH, "rb") as f:
        sent_tokens_list = pickle.load(f)
    # 캐시 생성 이후 추가된 불용어를 재적용 (kiwi 재호출 없이 필터만)
    extra_stop = STOPWORDS  # synonym_stopword.py에서 import된 최신 STOPWORDS 사용
    sent_tokens_list = [
        [[tok for tok in sent if tok not in extra_stop] for sent in sent_list
         if [tok for tok in sent if tok not in extra_stop]]
        for sent_list in sent_tokens_list
    ]
    print("캐시에서 로드 완료 (추가 불용어 재적용)")
else:
    sent_tokens_list = [
        sent_tokenize(text) for text in tqdm(df["text_cleaned"], desc="문장 토큰화")
    ]
    with open(CACHE_PATH, "wb") as f:
        pickle.dump(sent_tokens_list, f)
    print("토큰화 완료 및 캐시 저장")

df = df.copy()
df["sent_tokens"] = sent_tokens_list

In [ ]:
global_freq = Counter()
for sent_list in df["sent_tokens"]:
    for sent in sent_list:
        # 캐시 생성 이후 추가된 불용어까지 이 단계에서 재필터
        global_freq.update(w for w in sent if w not in STOPWORDS)

top_n = 150
vocab = {word for word, _ in global_freq.most_common(top_n)}

print(f"총 어휘 크기: {len(global_freq):,}개 → Top {top_n} 선정")
print("\nTop 20 단어:")
for word, cnt in global_freq.most_common(20):
    print(f"  {word}: {cnt:,}")

In [ ]:
def build_cooc(sent_token_series, vocab: set) -> Counter:
    """문장 내 어휘 단어 쌍의 공출현 Counter 반환"""
    cooc = Counter()
    for sent_list in sent_token_series:
        for sent in sent_list:
            # vocab은 이미 STOPWORDS 필터된 global_freq 기반이므로 자동 제외됨
            in_vocab = sorted({w for w in sent if w in vocab})
            for w1, w2 in combinations(in_vocab, 2):
                cooc[(w1, w2)] += 1
    return cooc

global_cooc = build_cooc(df["sent_tokens"], vocab)

cooc_df = pd.DataFrame(
    [(w1, w2, cnt) for (w1, w2), cnt in global_cooc.items()],
    columns=["word1", "word2", "weight"]
).sort_values("weight", ascending=False)

cooc_df.to_csv(output_dir / "cooc_global.csv", index=False, encoding="utf-8-sig")
print(f"전체 엣지 수: {len(cooc_df):,}")
print("\n상위 10개 공출현 쌍:")
print(cooc_df.head(10).to_string(index=False))

In [12]:
yearly_cooc = {}
for year, grp in df.groupby("year"):
    yearly_cooc[year] = build_cooc(grp["sent_tokens"], vocab)
    df_y = pd.DataFrame(
        [(w1, w2, cnt) for (w1, w2), cnt in yearly_cooc[year].items()],
        columns=["word1", "word2", "weight"]
    ).sort_values("weight", ascending=False)
    df_y.to_csv(output_dir / f"cooc_{year}.csv", index=False, encoding="utf-8-sig")
    print(f"{year}년: 엣지 {len(df_y):,}개")

print(f"\n연도별 처리 완료: {sorted(yearly_cooc.keys())}")

2016년: 엣지 11,103개
2017년: 엣지 11,102개
2018년: 엣지 11,081개
2019년: 엣지 11,043개
2020년: 엣지 11,044개
2021년: 엣지 11,135개
2022년: 엣지 11,058개
2023년: 엣지 11,011개
2024년: 엣지 10,978개
2025년: 엣지 10,990개

연도별 처리 완료: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [ ]:
def make_graph(cooc: Counter, min_weight: int = 1, top_k: int = None) -> nx.Graph:
    """공출현 Counter → NetworkX 그래프. top_k 지정 시 상위 K개 엣지만 유지."""
    G = nx.Graph()
    pairs = [(cnt, w1, w2) for (w1, w2), cnt in cooc.items() if cnt >= min_weight]
    if top_k is not None:
        pairs = sorted(pairs, reverse=True)[:top_k]
    for cnt, w1, w2 in pairs:
        G.add_edge(w1, w2, weight=cnt)
    G.remove_nodes_from(list(nx.isolates(G)))
    return G

# 전체 기간: 상위 300 엣지 (density ≈ 2.7%)
G_global = make_graph(global_cooc, top_k=300)
nx.write_gexf(G_global, str(output_dir / "network_global.gexf"))

# 연도별: 상위 150 엣지 (density ≈ 1.3%)
for year, cooc in yearly_cooc.items():
    G_year = make_graph(cooc, top_k=150)
    nx.write_gexf(G_year, str(output_dir / f"network_{year}.gexf"))

print(f"[전체] 노드: {G_global.number_of_nodes()}, 엣지: {G_global.number_of_edges()}, 밀도: {nx.density(G_global):.4f}")
print(f"GEXF 파일 저장 완료: {output_dir}")

### Step 5. 빈도 분석 시각화 (RQ1)

In [ ]:
# 셀 V1 — 전체 기간 Top 20 막대그래프
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

top20 = global_freq.most_common(20)
words, counts = zip(*top20)

plt.figure(figsize=(12, 6))
plt.bar(words, counts, color='steelblue')
plt.title('청년 담론 전체 기간 빈출 단어 Top 20 (2016–2025)', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.ylabel('빈도')
plt.tight_layout()
plt.savefig(output_dir / 'fig_top20_bar.png', dpi=150)
plt.show()
print(f"저장 완료: {output_dir / 'fig_top20_bar.png'}")

In [ ]:
# 셀 V2 — 연도별 Top 30 단어 상대빈도 Heatmap
import seaborn as sns

# 연도별 빈도 계산 (STOPWORDS 필터 적용)
yearly_freq = {}
for year, grp in df.groupby("year"):
    freq = Counter()
    for sent_list in grp["sent_tokens"]:
        for sent in sent_list:
            freq.update(w for w in sent if w not in STOPWORDS)
    yearly_freq[year] = freq

# Top 30 단어 × 연도 매트릭스
top30_words = [w for w, _ in global_freq.most_common(30)]
heat_df = pd.DataFrame(
    {year: [yearly_freq[year].get(w, 0) for w in top30_words] for year in sorted(yearly_freq)},
    index=top30_words
)

# 연도별 정규화 (연도 간 기사 수 차이 보정, 단위: ‰)
heat_norm = heat_df.div(heat_df.sum(axis=0), axis=1) * 1000

plt.figure(figsize=(14, 10))
sns.heatmap(heat_norm, cmap='YlOrRd', annot=True, fmt='.1f', linewidths=0.3)
plt.title('연도별 주요 단어 상대빈도 Heatmap (‰)', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'fig_yearly_heatmap.png', dpi=150)
plt.show()
print(f"저장 완료: {output_dir / 'fig_yearly_heatmap.png'}")

In [ ]:
# 셀 V3 — 워드클라우드
from wordcloud import WordCloud

wc = WordCloud(
    font_path='C:/Windows/Fonts/malgun.ttf',
    background_color='white',
    width=1200,
    height=600,
    max_words=100
)
wc.generate_from_frequencies(dict(global_freq))

plt.figure(figsize=(15, 7))
plt.imshow(wc, interpolation='bilinear')
plt.axis('off')
plt.title('청년 담론 워드클라우드 (2016–2025)', fontsize=14)
plt.tight_layout()
plt.savefig(output_dir / 'fig_wordcloud.png', dpi=150)
plt.show()
print(f"저장 완료: {output_dir / 'fig_wordcloud.png'}")

### Step 6. 의미연결망 분석 (RQ2)

In [ ]:
# 셀 N1 — 중심성 3종 계산 및 CSV 저장
import networkx as nx

deg = nx.degree_centrality(G_global)
btw = nx.betweenness_centrality(G_global, weight='weight')
eig = nx.eigenvector_centrality(G_global, weight='weight', max_iter=1000)

centrality_df = pd.DataFrame({'degree': deg, 'betweenness': btw, 'eigenvector': eig})
centrality_df = centrality_df.sort_values('degree', ascending=False)
centrality_df.to_csv(output_dir / 'centrality_global.csv', encoding='utf-8-sig')

print("=== 연결정도 중심성 Top 10 ===")
print(centrality_df['degree'].nlargest(10).to_string())
print("\n=== 매개 중심성 Top 10 ===")
print(centrality_df['betweenness'].nlargest(10).to_string())
print("\n=== 위세 중심성 Top 10 ===")
print(centrality_df['eigenvector'].nlargest(10).to_string())
print(f"\n저장 완료: {output_dir / 'centrality_global.csv'}")

In [ ]:
# 셀 N2 — 네트워크 기본 지표
print(f"노드 수: {G_global.number_of_nodes()}")
print(f"엣지 수: {G_global.number_of_edges()}")
print(f"밀도: {nx.density(G_global):.4f}")
print(f"포괄성: {G_global.number_of_nodes() / len(vocab):.3f}")

# 최대 연결 성분에서 평균 연결거리 계산
giant = G_global.subgraph(max(nx.connected_components(G_global), key=len)).copy()
print(f"최대 연결 성분 노드 수: {giant.number_of_nodes()}")
print(f"평균 연결거리: {nx.average_shortest_path_length(giant):.3f}")
print(f"군집계수(평균): {nx.average_clustering(G_global):.4f}")

### Step 7. 연도별 비교 분석 (RQ3)

In [ ]:
# 셀 Y1 — 특정 키워드 연도별 빈도 추이 선 그래프
# yearly_freq는 셀 V2에서 이미 계산됨

keywords = ['일자리', 'MZ세대', '공정', '주거', '저출산', '코로나', '세대갈등']

trend_df = pd.DataFrame(
    {w: [yearly_freq[y].get(w, 0) for y in sorted(yearly_freq)] for w in keywords},
    index=sorted(yearly_freq)
)

# 연도별 총 빈도로 정규화 (만분율)
total_by_year = [sum(yearly_freq[y].values()) for y in sorted(yearly_freq)]
trend_norm = trend_df.div(total_by_year, axis=0) * 10000

trend_norm.plot(figsize=(14, 6), marker='o')
plt.title('주요 단어 연도별 빈도 추이 (만분율)', fontsize=14)
plt.xlabel('연도')
plt.ylabel('빈도 (‱)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(output_dir / 'fig_keyword_trend.png', dpi=150)
plt.show()
print(f"저장 완료: {output_dir / 'fig_keyword_trend.png'}")

In [ ]:
# 셀 Y2 — 연도별 매개 중심성 순위 변화 (Bump Chart)
# centrality_df는 셀 N1에서 이미 계산됨

yearly_btw = {}
for year, cooc in yearly_cooc.items():
    G_y = make_graph(cooc, min_weight=2)
    yearly_btw[year] = nx.betweenness_centrality(G_y, weight='weight')

top10_words = centrality_df['betweenness'].nlargest(10).index.tolist()

rank_df = pd.DataFrame(
    {year: pd.Series(yearly_btw[year]).rank(ascending=False) for year in sorted(yearly_btw)}
).loc[top10_words]

fig, ax = plt.subplots(figsize=(14, 6))
for word in top10_words:
    ax.plot(sorted(yearly_btw.keys()), rank_df.loc[word], marker='o', label=word)
    ax.annotate(word, xy=(sorted(yearly_btw.keys())[-1], rank_df.loc[word].iloc[-1]),
                xytext=(3, 0), textcoords='offset points', va='center', fontsize=8)

ax.invert_yaxis()
ax.set_title('주요 단어 매개 중심성 순위 변화 (Bump Chart)', fontsize=14)
ax.set_xlabel('연도')
ax.set_ylabel('순위 (낮을수록 높음)')
ax.legend(loc='upper left', bbox_to_anchor=(1, 1), fontsize=8)
plt.tight_layout()
plt.savefig(output_dir / 'fig_bump_chart.png', dpi=150)
plt.show()
print(f"저장 완료: {output_dir / 'fig_bump_chart.png'}")